In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, VBox, interactive_output

# 1. Create test signal N = 32 in normal order
np.random.seed(42)
sample_values = np.round(np.random.uniform(1, 10, 32) + 1j * np.random.uniform(0, 5, 32), 2)
N = len(sample_values)
num_stages = int(np.log2(N))

# 2. Bit-reversal helper function (used for DIF output ordering)
def bit_reverse_indices(n_pts):
    bits = int(np.log2(n_pts))
    rev = np.zeros(n_pts, dtype=int)
    for i in range(n_pts):
        r = 0
        for j in range(bits):
            if (i & (1 << j)) != 0:
                r |= (1 << (bits - 1 - j))
        rev[i] = r
    return rev

# 3. Stage simulation for Decimation in Frequency (DIF)
stage_data = []
stage_math = []

curr = sample_values.astype(complex)
stage_data.append(curr.copy())
stage_math.append("Stage 0: Initial Normal Input State\n- Standard sequential order (Length L = N).")

for s in range(1, num_stages + 1):
    L = N >> (s - 1)  # Block length halves each stage: 32, 16, 8, 4, 2
    half_L = L >> 1
    nxt = np.zeros_like(curr)
    
    math_log = f"Stage {s} DIF Simulation (Block Length L = {L}, Half-Length = {half_L}):\n"
    math_log += f"Butterfly: Addition/Subtraction first, then Twiddle multiplication.\n"
    math_log += "-" * 55 + "\n"
    
    for i in range(0, N, L):
        math_log += f"[Block start index {i}]\n"
        for j in range(half_L):
            angle = -2 * np.pi * j / L
            w = np.exp(1j * angle)
            
            u = curr[i + j]
            v = curr[i + j + half_L]
            
            # DIF Butterfly core operations
            out_u = u + v
            out_v = (u - v) * w
            
            nxt[i + j]             = out_u
            nxt[i + j + half_L]     = out_v
            
            w_str = f"{w.real:.2f}{'+' if w.imag>=0 else ''}{w.imag:.2f}j" if abs(w.imag)>1e-4 else f"{w.real:.2f}"
            math_log += f"  j={j} (W={w_str}):\n"
            math_log += f"    X({i+j})       = ({u.real:.2f}+{u.imag:.2f}j) + ({v.real:.2f}+{v.imag:.2f}j) = {out_u.real:.2f}+{out_u.imag:.2f}j\n"
            math_log += f"    X({i+j+half_L}) = [({u.real:.2f}+{u.imag:.2f}j) - ({v.real:.2f}+{v.imag:.2f}j)] * ({w_str}) = {out_v.real:.2f}+{out_v.imag:.2f}j\n"
            
    curr = nxt
    stage_data.append(curr.copy())
    stage_math.append(math_log)

# Finally, apply bit-reversal to the last stage output for correct final spectrum order if needed,
# or keep the natural flow. Let's store the stages cleanly.

color_list = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

def plot_interactive_dif_32(stage):
    plt.close('all')
    fig = plt.figure(figsize=(18, 9.5))
    gs = fig.add_gridspec(2, 2, width_ratios=[2.2, 1], height_ratios=[1, 1], hspace=0.32, wspace=0.15)
    
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[1, 0])
    ax_math = fig.add_subplot(gs[:, 1])
    
    n_pts = np.arange(N)
    
    # --- TOP PLOT: Input state ---
    mags_in = np.abs(stage_data[max(0, stage - 1)])
    ax1.set_title(f"Stage {stage} Input (DIF Flow)", fontsize=11, fontweight='bold', pad=8)
    
    max_val_in = np.max(mags_in) if np.max(mags_in) > 0 else 1.0
    ax1.set_ylim(0, max_val_in * 1.35)
    
    for i in range(N):
        ax1.stem([i], [mags_in[i]], linefmt='#1f77b4', markerfmt='o', basefmt='k-')
        ax1.annotate(f"{mags_in[i]:.1f}", (i, mags_in[i]),
                     textcoords="offset points", xytext=(0, 14),
                     ha='center', fontsize=6.5, fontweight='bold', color='#1f77b4')
        
    ax1.set_ylabel("Magnitude |X[k]|", fontsize=10)
    ax1.grid(True, linestyle='--', alpha=0.5)
    ax1.set_xticks(n_pts)
    ax1.set_xticklabels([str(i) for i in n_pts], fontsize=8)

    # --- BOTTOM PLOT: Output state ---
    mags_out = np.abs(stage_data[stage])
    ax2.set_title(f"Stage {stage} Output (DIF Decimation)", fontsize=11, fontweight='bold', pad=8)
    
    max_val_out = np.max(mags_out) if np.max(mags_out) > 0 else 1.0
    ax2.set_ylim(0, max_val_out * 1.35)
    
    for i in range(N):
        ax2.stem([i], [mags_out[i]], linefmt='#ff7f0e', markerfmt='s', basefmt='k-')
        ax2.annotate(f"{mags_out[i]:.1f}", (i, mags_out[i]),
                     textcoords="offset points", xytext=(0, 14),
                     ha='center', fontsize=6.5, fontweight='bold', color='#ff7f0e')
        
    ax2.set_xlabel("Index ($k$)", fontsize=10)
    ax2.set_ylabel("Magnitude |X[k]|", fontsize=10)
    ax2.grid(True, linestyle='--', alpha=0.5)
    ax2.set_xticks(n_pts)
    ax2.set_xticklabels([str(i) for i in n_pts], fontsize=8)

    # --- RIGHT PANEL: Logs ---
    ax_math.axis('off')
    math_text = stage_math[stage]
    ax_math.text(0.02, 0.98, math_text, fontsize=7.5, family='monospace', va='top', ha='left',
                 bbox=dict(boxstyle='round,pad=1', facecolor='whitesmoke', edgecolor='lightgray', alpha=0.95))

    plt.show()

# Widget binding
stage_slider = IntSlider(value=1, min=1, max=num_stages, step=1, description='DIF Stage:')
out = interactive_output(plot_interactive_dif_32, {'stage': stage_slider})

display(VBox([stage_slider, out]))